# Kvartalsvis pro forma-resultatopgørelse med PROC COMPUTAB

## Resumé

Denne notebook opbygger en regional banks kvartalsvise pro forma-resultatopgørelse direkte ud fra månedlige hovedbogsdata ved hjælp af PROC COMPUTAB, SAS/ETS-proceduren til rapportskrivning med tabeller. Den fører hver måneds renteindtægter, renteudgifter, gebyrindtægter og driftsomkostninger over i den korrekte kalenderkvartalskolonne og bruger derefter rækkeprogrammeringsblokke til at beregne nettorenteindtægter, resultat før skat og nettoresultat samt en kolonneblok til at samle de fire kvartaler i en samlet årssum.

## Datakilder

Det enkelte syntetiske datasæt `bank_ledger` simulerer ét regnskabsår med månedlige regnskabsposter for en mellemstor lokal bank. Tolv månedlige observationer genereres inline med `call streaminit`/`rand`, så denne notebook er fuldt selvstændig.

| Variabel | Type | Beskrivelse |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Statusdato ved månedens udgang (jan.-dec.) |
| `loanint`  | Num | Renter og gebyrer optjent på udlånsporteføljen (USD i tusinder) |
| `secint`   | Num | Renter optjent på beholdningen af investeringsværdipapirer (USD i tusinder) |
| `depint`   | Num | Renter betalt på indlån (USD i tusinder) |
| `borrint`  | Num | Renter betalt på lån / FHLB-forskud (USD i tusinder) |
| `feeinc`   | Num | Ikke-renteindtægter (gebyr- og serviceindtægter) (USD i tusinder) |
| `salaries` | Num | Lønninger og personalegoder (USD i tusinder) |
| `occupancy`| Num | Lokale- og udstyrsomkostninger (USD i tusinder) |
| `othropex` | Num | Øvrige driftsomkostninger uden for renter (USD i tusinder) |
| `provision`| Num | Hensættelse til kredittab (USD i tusinder) |
| `taxrate`  | Num | Effektiv skattesats anvendt på resultat før skat |

# Kvartalsvis pro forma-resultatopgørelse med PROC COMPUTAB

Bankers økonomiafdelinger omdanner rutinemæssigt en månedlig hovedbog til en **kvartalsvis resultatopgørelse**, der viser nettorenteindtægter og bundlinjens nettoresultat. `PROC COMPUTAB` (SAS/ETS) er skabt netop til dette: den opsætter en programmerbar tabel, hvis **kolonner** er rapporteringsperioderne, og hvis **rækker** er linjeposterne i opgørelsen, og den lader dig beregne subtotaler med almindelig DATA step-logik i række- og kolonneblokke.

I denne notebook gør vi følgende:

1. Genererer ét regnskabsår med syntetiske månedlige hovedbogsdata for en lokal bank.
2. Fører hver måned over i sit kalenderkvartal og opbygger en kvartalsvis resultatopgørelse.
3. Beregner nettorenteindtægter, resultat før skat og nettoresultat i en **rækkeblok** og samler kvartalerne i en samlet årssum i en **kolonneblok**.
4. Genbruger `out=`-tabellen, så den beregnede opgørelse kan indgå i efterfølgende rapportering.

## Trin 1 — Generér syntetiske månedlige hovedbogsdata

Vi simulerer tolv observationer ved månedens udgang. Renteindtægter fra udlån og værdipapirer stiger svagt over året, indlåns- og låneomkostninger følger renteniveauet, og gebyrindtægter, driftsomkostninger og hensættelsen til kredittab har realistisk måned-til-måned-støj. `call streaminit` fastsætter frøet, så dataene er reproducerbare.

In [1]:
data bank_ledger;
   CALL streaminit(20240115);
   format stmtdate date9.;
   GØR month = 1 TIL 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      UDDATA;
   SLUT;
   BEHOLD stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
KØR;

PROCEDURE UDSKRIV data=bank_ledger noobs MÆRKAT;
   TITEL 'Synthetic Monthly Ledger — Community Bank FY2024';
   format loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
KØR;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## Trin 2 — Opbyg den kvartalsvise resultatopgørelse

Kernen i rapporten er `PROC COMPUTAB`-trinnet nedenfor.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definerer fire kvartalskolonner plus en kolonne for hele året.
- Den umærkede **inputblok** sætter den automatiske variabel **`_col_`** med `qtr(stmtdate)`, som fører hver månedlig observation over i den rette kvartalskolonne. Da inputtet som standard transponeres, lander hver hovedbogsvariabel i den række, der deler dens navn.
- **Rækkeblokken** `inc_stmt:` kører én gang pr. kolonne og beregner de afledte linjer — nettorenteindtægter, samlede ikke-renteudgifter, resultat før skat og nettoresultat — præcis som en revisor ville gøre det.
- **Kolonneblokken** `total:` kører én gang pr. række og lægger de fire kvartaler sammen i kolonnen `fy` (hele året).

`rows ... / ...`-sætningerne tilføjer titler, indrykning og stregelinjer (`ol` overstregning, `ul` understregning, `dul` dobbelt understregning), så outputtet læses som en rigtig resultatopgørelse.

In [2]:
TITEL 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

PROCEDURE computab data=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / format=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
KØR;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## Trin 3 — Genbrug COMPUTAB-outputdatasættet

`PROC COMPUTAB`-trinnet ovenfor skrev sin beregnede tabel til `qtr_income` via `out=`. Hver række i det datasæt er en linjepost i opgørelsen, og hver kolonnevariabel (`qtr1`-`qtr4`, `fy`) indeholder den beregnede værdi, så det kan indgå i efterfølgende rapportering. Nedenfor udskriver vi den samlede årskolonne for at bekræfte, at tallene stemmer.

In [3]:
TITEL 'COMPUTAB Output Dataset — Full-Year Column';

PROCEDURE UDSKRIV data=qtr_income MÆRKAT noobs;
   VARIABEL _row_ fy;
   format fy comma12.1;
   MÆRKAT _row_ = 'Line Item' fy = 'Full Year';
KØR;

TITEL;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Fortolkning af resultaterne

Pro forma-opgørelsen læses fra top til bund som en banks lovpligtige indberetning (call report): samlede renteindtægter minus samlede renteudgifter giver **nettorenteindtægter** — her omkring \$20,5 mio. for året — bankens primære indtjeningskilde. Ved at fratrække **hensættelsen til kredittab**, tillægge **gebyrindtægter** og modregne **ikke-renteudgifter** fås et resultat før skat på cirka \$9,0 mio., og anvendelse af den effektive skattesats på 21 % giver et **nettoresultat** tæt på \$7,1 mio. Kolonneblokken `total:` lægger de fire kvartaler sammen i kolonnen for hele året, så årstotalerne per konstruktion afstemmes til summen af kvartalerne — bekræftet i `out=`-datasættet, hvor årets samlede `netinc` på 7.081,6 svarer til summen af de fire kvartalstal.

Fordi alt beregnes inde i `PROC COMPUTAB`'s programmerbare række- og kolonneblokke, kræver det ingen ændring af rapportlogikken at indsætte en rigtig månedlig hovedbog — kun inputdatasættet ændres. `out=`-datasættet gør derefter den beregnede opgørelse tilgængelig til dashboards, trendanalyse eller automatisering af bestyrelsesmateriale.